# Generate canonical Vega datasets → `pb_demo.custom_gallery`

Loads the datasets the [Vega-Lite example gallery](https://vega.github.io/vega-lite/examples/)
uses (cars, seattle-weather, stocks, movies, barley, population, …) from the
`vega_datasets` package and writes each as a Delta table in `pb_demo.custom_gallery`.

Column names are normalized (spaces / punctuation → `_`) so they are clean to
reference from Vega-Lite `custom-vega-viz` widgets. Run top-to-bottom on a
Serverless notebook.

In [ ]:
%pip install vega_datasets

In [ ]:
dbutils.library.restartPython()

In [ ]:
import re
from vega_datasets import data
import pyspark.sql.functions as F

# Parameterized so a job can target any catalog.schema (defaults match the repo).
dbutils.widgets.text("catalog", "pb_demo")
dbutils.widgets.text("schema", "custom_gallery")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}"); spark.sql(f"USE SCHEMA {SCHEMA}")

def norm(c: str) -> str:
    c = re.sub(r"[^0-9a-zA-Z]+", "_", str(c)).strip("_")
    return c or "col"

def write_ds(name, pdf):
    pdf = pdf.copy()
    pdf.columns = [norm(c) for c in pdf.columns]
    # stringify any object/date columns that Spark cannot infer cleanly
    for col in pdf.columns:
        if pdf[col].dtype == "object":
            pdf[col] = pdf[col].astype(str)
    sdf = spark.createDataFrame(pdf)
    (sdf.write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{name}"))
    print(f"wrote {CATALOG}.{SCHEMA}.{name}")


In [ ]:
# (loader, table_name).  Wrapped so one bad dataset doesn't abort the run.
loaders = [
    (data.cars,        "cars"),
    (data.movies,      "movies"),
    (data.seattle_weather, "seattle_weather"),
    (data.stocks,      "stocks"),
    (data.sp500,       "sp500"),
    (data.barley,      "barley"),
    (data.population,  "population"),
    (data.disasters,   "disasters"),
    (data.wheat,       "wheat"),
    (data.us_employment, "us_employment"),
    (data.iowa_electricity, "iowa_electricity"),
    (data.co2_concentration, "co2_concentration"),
    (data.gapminder,   "gapminder"),
    (data.github,      "github"),
    (data.driving,     "driving"),
    (data.ohlc,        "ohlc"),
    (data.unemployment_across_industries, "unemployment_across_industries"),
    (data.jobs,        "jobs"),
    (data.birdstrikes, "birdstrikes"),
    (data.airports,    "airports"),
]
print("Generating canonical Vega datasets...")
for loader, name in loaders:
    try:
        write_ds(name, loader())
    except Exception as e:
        print(f"  SKIP {name}: {type(e).__name__}: {e}")
print("Done.")

In [ ]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}"))